# SWAMPE-JAX &rarr; phase-curve retrieval — full pipeline (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imalsky/SWAMPE-JAX/blob/master/retrieval/scripts/full_retrieval.ipynb)

This notebook runs the same end-to-end retrieval as `retrieval/scripts/run.sh`,
adapted from the JPL SLURM/conda cluster setup to a plain Colab runtime:

```
parameters -> SWAMPE-JAX (my_swampe) -> brightness-temperature map -> intensity map
           -> starry/jaxoplanet phase curve -> Gaussian likelihood
           -> BlackJAX adaptive tempered SMC -> posterior + diagnostic figures
```

Repo: <https://github.com/imalsky/SWAMPE-JAX> — see `retrieval/README.md` for the
science background (what `tau_rad`/`tau_drag` are physically, the noise models,
the priors, why SMC instead of plain MCMC).

**Before running:** for the `gpu`/`prod` presets, go to
`Runtime > Change runtime type` and pick a GPU. The `fast` preset is CPU-friendly
and is the right first run / sanity check (works on a free CPU or T4 runtime,
roughly 15-60 min depending on the runtime). The `gpu` preset is the full
paper-aligned retrieval (20-day spin-up, float64, 64-particle SMC swarm) and
takes roughly 1-2h on an A100 — that can exceed a free-tier Colab session, so use
Colab Pro/Pro+ (or interact with the tab periodically) if you select it.

Run the cells top to bottom, or set the config form below and `Runtime > Run all`.

## Step 0 — Configure the run

This only sets variables — nothing runs yet.

In [ ]:
#@title Run configuration { display-mode: "form" }

PRESET = "fast"  #@param ["fast", "gpu", "prod"]
#@markdown - `fast` -- ~2-day spin-up, float32, 24-particle SMC swarm. CPU-friendly smoke test (~15-60 min).
#@markdown - `gpu` -- the full paper-aligned retrieval: 20-day spin-up, float64, 64-particle SMC, photon noise. Needs a GPU runtime (~1-2h on an A100).
#@markdown - `prod` -- the package's 50-day production default (heaviest).

USE_X64 = "auto"  #@param ["auto", "0", "1"]
#@markdown `auto` lets the preset decide (gpu/prod default to float64, fast to float32).

OVERRIDES_JSON = ""  #@param {type:"string"}
#@markdown JSON of any `Config` field to override, e.g. `{"model_days": 5.0, "smc_num_particles": 32}`.

SEED = 7  #@param {type:"integer"}
SKIP_PLOTS = False  #@param {type:"boolean"}
RUN_SANITY_TESTS = True  #@param {type:"boolean"}
#@markdown Runs the fast pytest subset (CPU, ~1 min) before committing to a long run.

REPO_URL = "https://github.com/imalsky/SWAMPE-JAX.git"  #@param {type:"string"}
BRANCH = "master"  #@param {type:"string"}
REPO_DIR = "/content/SWAMPE-JAX"  #@param {type:"string"}
JAX_VERSION = ""  #@param {type:"string"}
#@markdown Leave blank (recommended) to keep Colab's preinstalled jax, which is already
#@markdown matched to this runtime's GPU driver. Only set an exact version (e.g. "0.6.2",
#@markdown what `run.sh` pins on the JPL cluster) if you need strict reproducibility and are
#@markdown prepared to debug CUDA-library compatibility yourself -- see the install cell below.

import json as _json

_overrides = _json.loads(OVERRIDES_JSON) if OVERRIDES_JSON.strip() else {}
_overrides.setdefault("seed", SEED)  # explicit "seed" in OVERRIDES_JSON wins over the field above

import os

os.environ["MY_SWAMPE_RETRIEVAL_PRESET"] = PRESET  # exported early so the backend check below can see it

print(f"preset={PRESET!r}  use_x64={USE_X64!r}  overrides={_overrides}")

## Step 1 — Check the runtime

In [ ]:
import subprocess

_nvidia = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
HAS_GPU = _nvidia.returncode == 0
if HAS_GPU:
    print(_nvidia.stdout.strip())
else:
    print("No GPU visible to this runtime.")

if PRESET in ("gpu", "prod") and not HAS_GPU:
    print(
        f"\nWARNING: preset={PRESET!r} is sized for a GPU and may take a very long "
        "time on CPU. Use Runtime > Change runtime type > GPU, or switch to "
        "PRESET='fast' in the config cell above and re-run it."
    )


## Step 2 — Clone the repository

In [ ]:
import os

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"{REPO_DIR} already exists; fetching latest {BRANCH}...")
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}/retrieval/scripts


## Step 3 — Install dependencies

Unlike `run.sh` (which starts from a bare SLURM conda env and must install an
exact `jax[cuda12]` pin), this notebook defaults to *keeping Colab's preinstalled
jax* on a GPU runtime — Colab already ships a jax/jaxlib/CUDA-plugin trio matched
to its own driver. Overriding that with an old exact pin is risky: JAX's CUDA
wheels (`jax-cuda12-plugin`) only floor-bound their `nvidia-*-cu12` sub-packages
(e.g. `nvidia-cudnn-cu12>=9.8`), so pip always resolves *today's* newest CUDA
libraries regardless of how old the JAX pin is — an old plugin binary against
today's CUDA libraries is a common source of `cuInit` failing with
`CUDA_ERROR_SHARED_OBJECT_INIT_FAILED` and a silent fall-back to CPU. Set
`JAX_VERSION` above only if you specifically need that reproducibility and are
prepared to debug it.

Unlike a bare HPC cluster, Colab's own `LD_LIBRARY_PATH` is left untouched here
— clearing it (a mitigation that makes sense on a cluster with a conflicting
`module load cuda`) turned out to break `cuInit` on Colab's own containerized
GPU setup, which otherwise works with no environment intervention at all.

In [ ]:
# NOTE: earlier revisions of this cell cleared LD_LIBRARY_PATH here, copying
# run.sh's SLURM-specific mitigation for a `module load cuda` conflict. On
# Colab that turned out to be actively harmful: the SAME cuInit/CUDA_ERROR_
# SHARED_OBJECT_INIT_FAILED (303) reproduced with BOTH a pinned old jax AND
# Colab's own stock jax, which normally just works on a GPU runtime with no
# intervention at all -- so the env mutation, not the jax version, was the
# actual cause. Colab's container likely needs its own LD_LIBRARY_PATH intact
# for the driver-level libcuda.so (never bundled in the jax wheel) to be
# discoverable. Leave it untouched here.
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
os.environ["PYTHONNOUSERSITE"] = "1"
os.environ["MPLBACKEND"] = "Agg"

if JAX_VERSION.strip():
    # Explicit pin requested. jax-cuda12-plugin's own nvidia-*-cu12 sub-deps are
    # floor-only (e.g. nvidia-cudnn-cu12>=9.8, no upper bound), so pip always
    # grabs TODAY's newest CUDA libraries no matter how old JAX_VERSION is -- an
    # old jax-cuda12-plugin binary can be ABI-incompatible with a much newer
    # nvidia-cublas-cu12/cudnn-cu12, which surfaces as cuInit failing with
    # CUDA_ERROR_SHARED_OBJECT_INIT_FAILED (303) and a silent fall-back to CPU.
    # The backend-check cell below will catch that loudly if it happens.
    if HAS_GPU:
        print(f"Installing jax[cuda12]=={JAX_VERSION} ...")
        !pip install -q --upgrade "jax[cuda12]=={JAX_VERSION}"
    else:
        print(f"Installing CPU-only jax=={JAX_VERSION} ...")
        !pip install -q --upgrade "jax=={JAX_VERSION}"
else:
    print("JAX_VERSION is blank -- keeping Colab's preinstalled jax (already matched "
          "to this runtime's GPU driver).")
    !python -c "import jax; print(f'preinstalled: jax {{jax.__version__}} | jaxlib {{jax.lib.__version__}}')"

# jaxoplanet's own requirements (equinox, jax, jaxlib) are all UNPINNED, so
# installing it normally (not --no-deps) cannot touch the jax pin above -- pip
# leaves an already-satisfied unpinned requirement alone. --no-deps here would
# silently skip equinox (a hard jaxoplanet dependency) on this fresh runtime
# and break the import.
!pip install -q --upgrade jaxoplanet
# Upper-bound blackjax: 1.4+ requires jax>=0.9.0/jaxlib>=0.9.0 (1.2.x/1.3.x
# only need jax/jaxlib>=0.4.16, satisfied by JAX_VERSION above). Unbounded,
# this silently drags jax/jaxlib up to satisfy blackjax while jax-cuda12-plugin
# (installed above, pinned to JAX_VERSION) is left behind -- the two then
# mismatch and JAX silently falls back to CPU.
!pip install -q --upgrade "blackjax>=1.2,<1.4"
!pip install -q --upgrade "corner>=2.2"


Verify the imports and the JAX backend (a separate `python -` subprocess, same as
`run.sh`, so JAX never gets imported into this notebook's own kernel process —
that matters once `run_smc.py` wants to own the whole GPU).

In [ ]:
%%bash
set -euo pipefail

python - <<'PY'
import jaxoplanet
from jaxoplanet.starry.light_curves import light_curve          # noqa: F401
from jaxoplanet.starry.surface import Surface                   # noqa: F401
import blackjax
from blackjax.smc import adaptive_tempered, resampling          # noqa: F401
import corner
print(f"jaxoplanet {jaxoplanet.__version__} | blackjax {getattr(blackjax,'__version__','?')} | corner {corner.__version__} : imports OK")
PY

python - <<'PY'
import os
import sys
import jax
backend = jax.default_backend()
print(f"jax {jax.__version__} | backend={backend} | devices={jax.devices()}")
if backend == "tpu":
    sys.exit("FATAL: TPU backend is not supported by this pipeline. "
              "Runtime > Change runtime type > GPU (or None for CPU).")
preset = os.environ.get("MY_SWAMPE_RETRIEVAL_PRESET", "fast")
if preset in ("gpu", "prod") and backend != "gpu":
    sys.exit(
        f"FATAL: PRESET={preset!r} needs a GPU backend but jax resolved to {backend!r}. "
        "This usually means a later pip install (e.g. blackjax) silently upgraded "
        "jax/jaxlib past what jax-cuda12-plugin was pinned to -- run "
        "'pip show jax jaxlib jax-cuda12-plugin' to check for a version mismatch, or "
        "Factory Reset Runtime and rerun from the top. Switch PRESET='fast' above if "
        "you want to proceed on CPU."
    )
PY


## Step 4 — Optional: quick correctness tests

Runs `retrieval/scripts/tests` (CPU, float32, ~1 min, deselecting the slow
end-to-end SMC test) before committing to a long run — catches a broken install
early instead of mid-run.

In [ ]:
if RUN_SANITY_TESTS:
    !python -m pytest tests -q -m "not slow"
else:
    print("Skipping sanity tests (RUN_SANITY_TESTS=False).")


## Step 5 — Run the retrieval

Sets the same env vars `run_smc.py` reads (see its docstring), then runs it as a
single process that owns the GPU for the duration of the SMC run — exactly like
`run.sh`.

In [ ]:
import time

os.environ["MY_SWAMPE_RETRIEVAL_PRESET"] = PRESET
if USE_X64 != "auto":
    os.environ["MY_SWAMPE_RETRIEVAL_USE_X64"] = USE_X64
else:
    os.environ.pop("MY_SWAMPE_RETRIEVAL_USE_X64", None)
if _overrides:
    os.environ["MY_SWAMPE_RETRIEVAL_OVERRIDES"] = _json.dumps(_overrides)
else:
    os.environ.pop("MY_SWAMPE_RETRIEVAL_OVERRIDES", None)

# One process owns the whole GPU for the run -- preallocate to avoid fragmentation
# through the vmapped scan. Only worth it for the bigger gpu/prod presets; the
# fast preset's default (no preallocation) is friendlier on a shared/free GPU.
if PRESET in ("gpu", "prod") and HAS_GPU:
    os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"
    os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.90")
    os.environ.setdefault("XLA_FLAGS", "--xla_gpu_triton_gemm_any=true")

t0 = time.time()
!python -u run_smc.py
print(f"\nrun_smc.py finished in {(time.time() - t0) / 60:.1f} min")


## Step 6 — Generate figures + summary

`plot_smc.py` / `make_dashboard.py` / `summarize_run.py` only read the `.npz`
bundles `run_smc.py` just wrote to `../data/` — they never re-run SWAMPE-JAX, so you
can re-run just this step (e.g. after restyling a plot) without redoing the SMC.

In [ ]:
if not SKIP_PLOTS:
    !python -u plot_smc.py
    !python -u make_dashboard.py
    !python -u summarize_run.py
else:
    print("Skipping plots (SKIP_PLOTS=True).")


## Step 7 — View results

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

plots_dir = Path("../plots")
data_dir = Path("../data")

dashboard = plots_dir / "results_dashboard.png"
if dashboard.exists():
    display(Image(filename=str(dashboard)))
else:
    print(f"No dashboard found at {dashboard} (did you skip plots?)")

summary = data_dir / "RETRIEVAL_SUMMARY.md"
if summary.exists():
    display(Markdown(summary.read_text()))
else:
    print(f"No summary found at {summary}")


In [ ]:
# Every other figure: phase-curve fit + residuals, 1-D posteriors with prior
# overlays, the tau_rad-tau_drag corner plot, SMC diagnostics, truth/posterior
# maps, and starry disk renders.
other_figs = sorted(p for p in plots_dir.glob("*.png") if p.name != "results_dashboard.png")
for p in other_figs:
    display(Markdown(f"**{p.name}**"))
    display(Image(filename=str(p)))


## Step 8 — Optional: save outputs

Colab runtimes are ephemeral — `../data/` and `../plots/` disappear when the
runtime recycles. Zip them for download and/or copy to Google Drive.

In [ ]:
#@title Save outputs { display-mode: "form" }
DOWNLOAD_ZIP = True  #@param {type:"boolean"}
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
DRIVE_DEST = "/content/drive/MyDrive/MY_SWAMPE_retrieval_outputs"  #@param {type:"string"}

import shutil
import zipfile

retrieval_root = Path("..").resolve()
zip_path = Path("/content/retrieval_outputs.zip")

if DOWNLOAD_ZIP or SAVE_TO_DRIVE:
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for sub in ("data", "plots"):
            for p in (retrieval_root / sub).rglob("*"):
                if p.is_file():
                    zf.write(p, arcname=str(p.relative_to(retrieval_root)))
    print(f"Wrote {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")

if DOWNLOAD_ZIP:
    try:
        from google.colab import files
        files.download(str(zip_path))
    except ImportError:
        print(f"Not running in Colab; find the archive at {zip_path}")

if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        Path(DRIVE_DEST).mkdir(parents=True, exist_ok=True)
        shutil.copy(zip_path, Path(DRIVE_DEST) / zip_path.name)
        print(f"Copied to {DRIVE_DEST}/{zip_path.name}")
    except ImportError:
        print("Not running in Colab; Drive mount unavailable.")


## Notes

- **Re-running just the plots**: tweak something in `plot_smc.py`/`make_dashboard.py`
  and re-run Step 6 + Step 7 only — they never re-run SWAMPE-JAX.
- **Only have a T4, not an A100**: the `gpu` preset is sized for an A100; lower
  `model_days` and/or `smc_num_particles` via `OVERRIDES_JSON` in Step 0, e.g.
  `{"model_days": 5.0, "smc_num_particles": 32}` (the forward model is already
  converged by ~5 days).
- **Session limits**: free Colab sessions can disconnect (~90 min idle, 12h hard
  cap). For the full `gpu`/`prod` presets, use Colab Pro/Pro+, or keep the tab
  active.
- **Many independent retrievals at once** (an SBC/coverage calibration study) is
  a different workload — see `coverage_study.py` and `retrieval/README.md`
  instead of this notebook.
- Full science background — what `tau_rad`/`tau_drag` are, the noise models, the
  priors, the `tau_rad`-`tau_drag` degeneracy — is in `retrieval/README.md` in the
  repo.